# 🎸 Justin Muir • Parametric NAM Offsite Cloud GPU Trainer
### 48kHz Audio DSP Lab — Train without needing an NVIDIA GPU locally!

This Google Colab notebook trains a **Parametric Neural Amp Modeler** profile across your 45 recorded re-amp takes.

#### The 3-Step Studio Workflow:
1. **Capture Locally**: Run your 45 calibration sweeps through your physical amp using `reamp_automation.py` or the REAPER ReaScript.
2. **Zip & Upload**: Zip your recorded WAV files along with `data.json` into `takes.zip` and upload below.
3. **Train & Download**: Colab crunches the neural weights on a free NVIDIA T4 GPU (~35–45 mins) and exports a standalone `.nam` binary file ready for your plugin!

## Step 1: Verify NVIDIA Cloud GPU Acceleration
Ensure your Colab runtime is set to GPU: **Runtime -> Change runtime type -> Hardware accelerator: T4 GPU**.

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"✅ NVIDIA GPU Online: {gpu_name} ({vram_gb:.1f} GB VRAM)")
else:
    print("❌ No GPU detected! Please go to Runtime -> Change runtime type -> select T4 GPU.")

## Step 2: Clone Parametric NAM Engine & Install Dependencies

In [ ]:
!git clone https://github.com/phillipmself/neural-amp-modeler-parametric.git
%cd neural-amp-modeler-parametric
!pip install -q lightning torch torchaudio soundfile scipy pandas
print("✅ Parametric NAM Engine & PyTorch Lightning dependencies installed successfully!")

## Step 3: Upload Your Zipped Takes (`takes.zip`)
Click **Choose Files** below and select your `takes.zip` (containing your rendered WAV files and `data.json`).

In [ ]:
import os
import zipfile
from google.colab import files

os.makedirs("./data", exist_ok=True)
print("📤 Please select your 'takes.zip' file to upload:")
uploaded = files.upload()

for fn in uploaded.keys():
    if fn.endswith(".zip"):
        with zipfile.ZipFile(fn, 'r') as z:
            z.extractall("./data/")
        print(f"✅ Successfully unzipped {fn} into ./data/")
    elif fn == "data.json":
        os.rename(fn, "./data/data.json")
        print("✅ Staged data.json into ./data/")

# Verify contents
data_files = os.listdir("./data")
print(f"📁 Staged {len(data_files)} files in ./data/ ready for training.")

## Step 4: Run Headless Batch Training
This trains the continuous Parametric WaveNet on the NVIDIA GPU using ESR + MRSTFT Loss.

In [ ]:
import os

os.makedirs("./output", exist_ok=True)

# Execute headless parametric training
!python -m nam.train.colab \
    ./data/data.json \
    configs/model_concat.json \
    configs/learning_concat.json \
    ./output/

print("🎉 Training execution completed!")

## Step 5: Download Your Standalone `.nam` Model
Automatically triggers a browser download of your compiled parametric amplifier binary.

In [ ]:
import glob
from google.colab import files

nam_files = glob.glob("./output/*.nam") + glob.glob("./output/**/*.nam", recursive=True)

if nam_files:
    target_model = nam_files[0]
    print(f"✅ Found compiled model: {target_model}")
    files.download(target_model)
    print("⬇️ Download initiated! You can now load this model in your NAM VST3/AU plugin.")
else:
    print("⚠️ No .nam file found in ./output. Check training logs above for details.")